In [2]:
import requests
import xmltodict
import json

class LiveEvidenceFetcher:
    def __init__(self):
        self.pubmed_search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
        self.pubmed_summary_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"
        self.clinical_trials_url = "https://clinicaltrials.gov/api/v2/studies"

    def fetch_clinical_trial(self, nct_id: str) -> dict:
        """Fetches structured trial details from ClinicalTrials.gov v2 API."""
        url = f"{self.clinical_trials_url}/{nct_id}"
        try:
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                data = response.json()
                protocol = data.get("protocolSection", {})
                
                # Extract clean clinical data points
                return {
                    "source": "ClinicalTrials.gov",
                    "id": nct_id,
                    "title": protocol.get("identificationModule", {}).get("officialTitle", "No Title Available"),
                    "status": protocol.get("statusModule", {}).get("overallStatus", "Unknown Status"),
                    "sponsor": protocol.get("sponsorCollaboratorsModule", {}).get("leadSponsor", {}).get("name", "Unknown"),
                    "conditions": protocol.get("conditionsModule", {}).get("conditions", []),
                    "interventions": [i.get("name") for i in protocol.get("armsInterventionsModule", {}).get("interventions", [])]
                }
            return {"error": f"Trial {nct_id} not found. Status code: {response.status_code}"}
        except Exception as e:
            return {"error": f"ClinicalTrials API call failed: {str(e)}"}

    def fetch_latest_pubmed(self, query: str, max_results: int = 3) -> list:
        """Queries PubMed for the latest research articles and returns clean summary details."""
        # Step 1: Search for relevant IDs
        search_params = {
            "db": "pubmed",
            "term": query,
            "retmode": "json",
            "retmax": max_results,
            "sort": "pub_date"  # Assures recency prioritization
        }
        try:
            search_res = requests.get(self.pubmed_search_url, params=search_params, timeout=10).json()
            id_list = search_res.get("esearchresult", {}).get("idlist", [])
            
            if not id_list:
                return [{"warning": "No recent articles found matching query."}]

            # Step 2: Fetch summaries for those IDs
            summary_params = {
                "db": "pubmed",
                "id": ",".join(id_list),
                "retmode": "json"
            }
            summary_res = requests.get(self.pubmed_summary_url, params=summary_params, timeout=10).json()
            uid_results = summary_res.get("result", {})

            articles = []
            for uid in id_list:
                if uid in uid_results:
                    art = uid_results[uid]
                    articles.append({
                        "source": "PubMed",
                        "pmid": uid,
                        "title": art.get("title", "No Title"),
                        "journal": art.get("source", "Unknown Journal"),
                        "pub_date": art.get("pubdate", "Unknown Date"),
                        "authors": [auth.get("name") for auth in art.get("authors", [])[:3]]
                    })
            return articles
        except Exception as e:
            return [{"error": f"PubMed API call failed: {str(e)}"}]

# --- Execution Test Block ---
fetcher = LiveEvidenceFetcher()

print("--- TESTING LIVE API ROUTING INTEGRATION ---")
trial_info = fetcher.fetch_clinical_trial("NCT02296125")
print(f"\n[Precision Result] NCT02296125 Info:\n{json.dumps(trial_info, indent=2)}")

pubmed_info = fetcher.fetch_latest_pubmed("KRAS G12C non-small cell lung cancer", max_results=2)
print(f"\n[Recency Result] PubMed Latest Updates:\n{json.dumps(pubmed_info, indent=2)}")

--- TESTING LIVE API ROUTING INTEGRATION ---

[Precision Result] NCT02296125 Info:
{
  "source": "ClinicalTrials.gov",
  "id": "NCT02296125",
  "title": "A Phase III, Double-blind, Randomised Study to Assess the Safety and Efficacy of AZD9291 Versus a Standard of Care Epidermal Growth Factor Receptor Tyrosine Kinase Inhibitor as First Line Treatment in Patients With Epidermal Growth Factor Receptor Mutation Positive, Locally Advanced or Metastatic Non Small Cell Lung Cancer.",
  "status": "COMPLETED",
  "sponsor": "AstraZeneca",
  "conditions": [
    "Locally Advanced or Metastatic EGFR Sensitising Mutation Positive Non Small Cell Lung Cancer"
  ],
  "interventions": [
    "AZD9291 80 mg/40 mg + placebo",
    "Placebo Erlotinib 150/100mg",
    "Placebo Gefitinib 250 mg",
    "Erlotinib 150/100 mg",
    "Gefitinib 250 mg",
    "Placebo AZD9291 80 mg/ 40 mg"
  ]
}

[Recency Result] PubMed Latest Updates:
[
  {
    "source": "PubMed",
    "pmid": "42190605",
    "title": "Sotorasib combin